# Image encoder benchmark — COSMOS Sérsic linear probe

Frozen `jwst_dino` teacher, linearly probed for the single-Sérsic fit labels
cross-matched from the COSMOS-Web photom catalog by row id: Sérsic index `n`
(→ log₁₀), effective radius `radius_sersic` (→ log₁₀ px), and axis ratio
`axratio_sersic`. Position angle is excluded (circular target).

**Sample.** The catalog fits, not the encoder, set the ceiling, so one scalar
threshold τ is shared by all three labels. Each catalog `*_err` is put on a
common scale as a fractional error in dex,

    σ_dex(x) = err(x) / (|x| · ln 10)

and a source is kept when the *worst* of its three σ_dex is under τ. Here
**τ = 0.05 dex** (≈12% relative error). The last cell scans τ to show that tightening it raises $R^2$ while the target variances stay flat, so the gain comes from removing label noise rather than from narrowing the label range. Two non-error cuts are applied as well: `R_e ≥ MIN_REFF_PX`
(resolved), and *railed* fits removed — parameters pinned at the fitter bounds
carry tiny formal errors, so no error threshold catches them. One τ ⇒ one sample
⇒ the three targets are probed on identical rows and their R² are comparable.

**Readout & probe.** Embedding is `concat(CLS, patch-mean)`. The probe is
an `L1` (lasso) probe (α by internal CV, no training steps), which measures
whether the label is linearly readable from the frozen features.

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
import warnings; warnings.simplefilter('ignore')

from model.jwst_dino import load_teacher_backbone
from data.augmentations import AsinhStretch

ROOT   = '~/ssl_outthere/data/image'
CKPT   = '/home/yacheng/ssl_outthere/encoder_image/jwst_dino/outputs/jwst_dino_ps6_st3/version_6/checkpoints/last.ckpt'
PHOTOM = '../../data/survey/cosmos_2025/COSMOSWeb_mastercatalog_v1_photom_primary.fits'
POSTER_DIR = '/home/yacheng/ssl_outthere/poster_figs'
os.makedirs(POSTER_DIR, exist_ok=True)

MIN_REFF_PX = 3.0       # resolved-source floor
TAU_DEX     = 0.05      # unified cut: max_j sigma_dex(label_j) <= TAU_DEX (~12% rel err)
TAU_SCAN    = [0.01, 0.02, 0.05, 0.1]   # label-quality scan (last cell)
MAX_SAMPLES = 30000     # cap the clean sample for speed (-1 = all)
TEST_FRAC   = 0.5
SEED        = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Regression targets (angle excluded): (photom column, plot label, log10 target?)
TARGETS = [('sersic',         r'$\log_{10} n$',         True),
           ('radius_sersic',  r'$\log_{10}\,R_e$ [px]', True),
           ('axratio_sersic', r'axis ratio $q$',        False)]

# Fitter bounds; a value pinned at a bound is a failed fit, not a measurement.
RAIL_BOUNDS = {'sersic': (0.31, 8.30), 'axratio_sersic': (0.05, 0.95)}
print('device:', DEVICE, ' tau_dex:', TAU_DEX)

In [ ]:
# frozen teacher backbone (crop_size comes from the checkpoint's training config)
net  = load_teacher_backbone(CKPT, DEVICE)
crop = net.crop_size
print('crop_size:', crop)

In [ ]:
DEG_TO_PIX = 3600 * 1000 / 30.0   # radius_sersic[deg] -> px at 30 mas/pix
LN10 = np.log(10)


class CosmosSersicDataset(Dataset):
    """COSMOS f150w cutouts paired with clean single-Sersic fit labels.

    id cross-match: image_index `id` = photom row index. Labels are stacked
    (N, 3) in `targets` order. A source is kept only if it is resolved, not
    railed, and its worst per-label sigma_dex = err/(|value|*ln10) is <= tau_dex.
    """

    def __init__(self, root, photom_catalog, targets, rail_bounds, filter='f150w',
                 crop_size=72, min_reff_px=3.0, tau_dex=0.05, max_samples=-1,
                 Q=20.0, scale=1.0, seed=42, verbose=True):
        self.root = os.path.expandvars(os.path.expanduser(root))
        self.center_crop = transforms.CenterCrop(crop_size)
        self.stretch = AsinhStretch(scale=scale, Q=Q, return_channel_pos=0)
        self.shards = {}

        index = Table.read(os.path.join(self.root, f'image_index_cosmos_{filter}.fits'))
        ids       = np.asarray(index['id'], np.int64)
        rel_path  = np.asarray(index['rel_path']).astype(str)
        local_idx = np.asarray(index['local_idx'], np.int64)

        ph = fits.open(os.path.expandvars(os.path.expanduser(photom_catalog)), memmap=True)[1].data
        get = lambda c: np.asarray(ph[c], np.float64)[ids]
        val = {'sersic': get('sersic'), 'radius_sersic': get('radius_sersic') * DEG_TO_PIX,
               'axratio_sersic': get('axratio_sersic')}
        err = {'sersic': get('sersic_err'), 'radius_sersic': get('radius_sersic_err') * DEG_TO_PIX,
               'axratio_sersic': get('axratio_sersic_err')}
        cols = [c for c, *_ in targets]

        resolved = (np.isfinite(val['radius_sersic']) & (val['radius_sersic'] >= min_reff_px)
                    & np.isfinite(val['sersic']) & np.isfinite(val['axratio_sersic'])
                    & (val['axratio_sersic'] > 0))
        railed = np.zeros(len(ids), bool)
        for c, (lo, hi) in rail_bounds.items():
            railed |= (val[c] <= lo) | (val[c] >= hi)

        sigma_dex = np.stack([err[c] / (np.abs(val[c]) * LN10) for c in cols], 1)
        usable = resolved & ~railed & np.isfinite(sigma_dex).all(1)
        clean = usable & (np.where(usable, sigma_dex.max(1), np.inf) <= tau_dex)

        keep = np.where(clean)[0]
        rng = np.random.default_rng(seed)
        if 0 < max_samples < len(keep):
            keep = rng.choice(keep, size=max_samples, replace=False)

        labels = np.stack([np.log10(val[c]) if log else val[c]
                           for c, _, log in targets], 1).astype(np.float32)
        self._samples = [(rel_path[i], local_idx[i]) for i in keep]
        self._labels  = labels[keep]
        self.sigma_dex_kept = sigma_dex[keep]
        if verbose:
            print(f'CosmosSersic [{filter}] - {len(ids)} cutouts, {resolved.sum()} resolved, '
                  f'{railed.sum()} railed, {clean.sum()} clean (tau<={tau_dex}), {len(keep)} used')

    def _shard(self, rp):
        if rp not in self.shards:
            self.shards[rp] = np.load(os.path.join(self.root, rp), mmap_mode='r')
        return self.shards[rp]

    def __len__(self):
        return len(self._samples)

    def __getitem__(self, i):
        rp, li = self._samples[i]
        img = np.nan_to_num(self._shard(rp)[li].astype(np.float32))[None]   # (1, H, W)
        img = self.center_crop(torch.from_numpy(img)).numpy()
        return torch.from_numpy(self.stretch(img)), self._labels[i]


ds = CosmosSersicDataset(ROOT, PHOTOM, TARGETS, RAIL_BOUNDS, crop_size=crop,
                         min_reff_px=MIN_REFF_PX, tau_dex=TAU_DEX,
                         max_samples=MAX_SAMPLES, seed=SEED)
loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)
print(f'median label sigma_dex per target: {np.median(ds.sigma_dex_kept, 0).round(4)}')

In [ ]:
# extract embeddings: concat(CLS, patch-mean)
@torch.no_grad()
def extract(net, loader, device):
    embs, labels = [], []
    for imgs, ys in tqdm(loader, desc='embeddings'):
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16,
                            enabled=device.type == 'cuda'):
            out = net(imgs.to(device))
        embs.append(torch.cat([out['cls'], out['patch'].mean(1)], 1).float().cpu().numpy())
        labels.append(np.asarray(ys))
    return np.concatenate(embs), np.concatenate(labels)

X, Y = extract(net, loader, DEVICE)
print('embeddings:', X.shape, ' labels:', Y.shape)

In [ ]:
# linear probe: standardize -> L1 (lasso), one head per target
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

def snmad(y, yp):
    d = yp - y
    return 1.4826 * np.median(np.abs(d - np.median(d)))

def l1_fit(Ztr, t):
    # alpha comes from LassoCV's data-adaptive path (an int requests that many
    # alphas spanning [1e-3, 1] x alpha_max); `edge` flags a solution that ran to
    # the end of that path, where the fit is no longer meaningfully penalised.
    m = LassoCV(alphas=25, cv=3, max_iter=200_000, tol=1e-3, selection="random",
                random_state=SEED, n_jobs=-1).fit(np.asarray(Ztr, np.float64), t)
    lo, hi = m.alphas_.min(), m.alphas_.max()
    return m, "FLOOR" if m.alpha_ <= lo * 1.001 else "CEIL" if m.alpha_ >= hi * 0.999 else "ok"

Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=TEST_FRAC, random_state=SEED)
scaler = StandardScaler().fit(Xtr)
Ztr, Zte = scaler.transform(Xtr), scaler.transform(Xte)

preds, PROBE = np.zeros_like(Yte), {}
hdr = f'{"target":16s} {"N":>6s} {"std":>7s} {"R2":>7s} {"sNMAD":>7s} {"MAE":>7s} {"RMSE":>7s} {"alpha":>9s} {"edge":>6s} {"act":>7s}'
print(f'=== Sersic linear probe  (tau<={TAU_DEX} dex, {len(Xtr)} train / {len(Xte)} test) ===')
print(hdr); print('-' * len(hdr))
for j, (col, _, _) in enumerate(TARGETS):
    reg, edge = l1_fit(Ztr, Ytr[:, j])
    preds[:, j] = reg.predict(Zte)
    m = dict(n=len(Yte), std=float(Yte[:, j].std()),
             r2=r2_score(Yte[:, j], preds[:, j]), snmad=snmad(Yte[:, j], preds[:, j]),
             mae=mean_absolute_error(Yte[:, j], preds[:, j]),
             rmse=float(np.sqrt(np.mean((Yte[:, j] - preds[:, j]) ** 2))),
             alpha=float(reg.alpha_), edge=edge,
             n_active=int((reg.coef_ != 0).sum()), n_feat=int(Ztr.shape[1]))
    PROBE[col] = m
    print(f'{col:16s} {m["n"]:6d} {m["std"]:7.3f} {m["r2"]:7.3f} {m["snmad"]:7.3f} '
          f'{m["mae"]:7.3f} {m["rmse"]:7.3f} {m["alpha"]:9.3e} {m["edge"]:>6s} '
          f'{m["n_active"]:4d}/{m["n_feat"]:d}')

In [ ]:
# pred-vs-true, NeurIPS poster style (matches LowResPT/neurips_spectrum_bench)
POSTER_RC = {
    'font.size': 14, 'axes.titlesize': 16, 'axes.labelsize': 14,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
}
mpl.rcParams.update(POSTER_RC)
# one colour per target, in TARGETS order
COLORS = ['royalblue', 'darkorange', 'seagreen']

fig, axes = plt.subplots(1, len(TARGETS), figsize=(6 * len(TARGETS), 5.6))
for j, (ax, (col, lbl, _)) in enumerate(zip(axes, TARGETS)):
    color = COLORS[j]
    yt, yp = Yte[:, j], preds[:, j]
    lo, hi = np.percentile(np.concatenate([yt, yp]), [0.5, 99.5])
    pad = 0.05 * (hi - lo)
    ax.scatter(yt, yp, s=6, alpha=0.25, edgecolors='none', color=color, rasterized=True)
    ax.plot([lo, hi], [lo, hi], '--', lw=2, color='black', alpha=0.4, label='1:1')
    m = PROBE[col]
    ax.text(0.97, 0.03,
            rf"$R^2$={m['r2']:.3f}   $\sigma_{{\rm NMAD}}$={m['snmad']:.3f}",
            transform=ax.transAxes, ha='right', va='bottom', fontsize=13,
            color=color, fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.7, pad=3.0, edgecolor='none'))
    ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlabel(f'{lbl}  (catalog)'); ax.set_ylabel(f'{lbl}  (predicted)')
    ax.legend(fontsize=12, loc='upper left')
    for sp in ax.spines.values():
        sp.set_linewidth(1.8)
    ax.tick_params(width=1.6)

fig.patch.set_alpha(0.0)
outp = os.path.join(POSTER_DIR, 'poster_image_sersic_probe')
fig.savefig(outp + '.png', transparent=False, dpi=300, bbox_inches='tight')
print(f'Saved -> {outp}.png')
plt.show()

In [ ]:
# label-quality scan: extract once at the loosest tau, then subset by tau.
# Samples are ordered as kept, so ds_scan.sigma_dex_kept aligns row-wise with X_s.
ds_scan = CosmosSersicDataset(ROOT, PHOTOM, TARGETS, RAIL_BOUNDS, crop_size=crop,
                              min_reff_px=MIN_REFF_PX, tau_dex=max(TAU_SCAN),
                              max_samples=MAX_SAMPLES, seed=SEED)
X_s, Y_s = extract(net, DataLoader(ds_scan, batch_size=128, shuffle=False,
                                   num_workers=8, pin_memory=True), DEVICE)
worst = ds_scan.sigma_dex_kept.max(1)

hdr = (f'{"tau":>6s} {"N":>7s} | ' +
       " | ".join(f'{c[:9]:>9s} R2  std' for c, *_ in TARGETS))
print(hdr); print('-' * len(hdr))
SCAN_TABLE = {}
for tau in sorted(TAU_SCAN):
    m = worst <= tau
    Xt, Yt = X_s[m], Y_s[m]
    itr, ite = train_test_split(np.arange(len(Xt)), test_size=TEST_FRAC, random_state=SEED)
    s = StandardScaler().fit(Xt[itr])
    Ztr_, Zte_ = s.transform(Xt[itr]), s.transform(Xt[ite])
    row, cells = {}, []
    for j, (col, _, _) in enumerate(TARGETS):
        p = l1_fit(Ztr_, Yt[itr, j])[0].predict(Zte_)
        r2, sd = r2_score(Yt[ite, j], p), float(Yt[ite, j].std())
        row[col] = dict(r2=r2, std=sd, snmad=snmad(Yt[ite, j], p), n=int(m.sum()))
        cells.append(f'{r2:12.3f} {sd:.3f}')
    SCAN_TABLE[tau] = row
    print(f'{tau:6.3f} {int(m.sum()):7d} | ' + " | ".join(cells))
print('\nR2 rises as tau tightens while std stays flat -> the gain is label-noise removal,')
print('not a narrowing of the target range.')